# 04 - Final Evaluation (Qwen3.5-4B V2 Fine-tuned)

Load best_mae_checkpoint, evaluate on 200 test items (seed=42), draw charts,
save results JSON. Run AFTER 03_train_v2 finishes.

Bám sát English Llama Section F: in `Memory footprint`, `print(fine_tuned_model)` repr (PeftModel wrapping LoraModel), display test[0], single-sample `model_predict` trước `set_seed(42)` + full evaluate.

In [ ]:
# Cell 1 - Imports + constants
import os, sys, json, torch
sys.path.insert(0, os.path.abspath("."))

from transformers import AutoTokenizer, AutoModelForCausalLM, set_seed
from peft import PeftModel

from utils.items_vn import load_items, DATASET_NAME
from utils.evaluator_vn import VnTester
from utils.training_utils import get_bnb_config, MAX_SEQ_LENGTH, MAX_NEW_TOKENS

BASE_MODEL   = "Qwen/Qwen3.5-4B-Base"
BEST_CKPT    = "outputs/qwen_v2/best_mae_checkpoint"
HUB_MODEL_ID = "SeanSunny/qwen3.5-4b-vn-pricer-v2"
RESULTS_PATH = "results/v2_results.json"
EVAL_SIZE    = 200

In [ ]:
# Cell 2 - Load base 4-bit + adapter (English Section F Cell 102-103)
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
tokenizer.pad_token = tokenizer.eos_token

base = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=get_bnb_config(),
    torch_dtype=torch.bfloat16,
    device_map="auto",
)
fine_tuned_model = PeftModel.from_pretrained(base, BEST_CKPT)
fine_tuned_model.eval()
print(f"Loaded adapter from: {BEST_CKPT}")
print(f"Memory footprint: {fine_tuned_model.get_memory_footprint() / 1e6:.1f} MB")

In [ ]:
# Cell 3 - Display fine_tuned_model architecture (English Cell 104)
# Expected: PeftModelForCausalLM wrapping LoraModel(Qwen3ForCausalLM) with lora_A/lora_B in each target module
print(fine_tuned_model)

In [ ]:
# Cell 4 - Load test items (seed=42, 200 items)
import random; random.seed(42)
all_test = load_items("test")
test_items = random.sample(all_test, min(EVAL_SIZE, len(all_test)))
print(f"Test items: {len(test_items)}")
print(f"Price range: {min(i.price for i in test_items):.0f}K - {max(i.price for i in test_items):.0f}K VND")

In [ ]:
# Cell 5 - Display test_items[0] (English Cell 101)
sample = test_items[0]
print(f"title:     {sample.title}")
print(f"price:     {sample.price}K VND ({sample.price_vnd:,} VND)")
print("prompt:")
print(sample.prompt)

In [ ]:
# Cell 6 - Predictor function (English Cell 105 pattern)
import re
def qwen_v2_predict(item) -> float:
    """Returns predicted price in K VND."""
    inputs = tokenizer(item.prompt, return_tensors="pt").to(fine_tuned_model.device)
    with torch.no_grad():
        out = fine_tuned_model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    prompt_len = inputs["input_ids"].shape[1]
    text = tokenizer.decode(out[0, prompt_len:], skip_special_tokens=True).strip()
    m = re.search(r"\d+\.?\d*", text)
    return float(m.group()) if m else 0.0


def qwen_v2_predict_raw(item) -> str:
    """Returns raw decoded text (no parsing) — for inspecting model output."""
    inputs = tokenizer(item.prompt, return_tensors="pt").to(fine_tuned_model.device)
    with torch.no_grad():
        out = fine_tuned_model.generate(
            **inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    prompt_len = inputs["input_ids"].shape[1]
    return tokenizer.decode(out[0, prompt_len:], skip_special_tokens=True)

In [ ]:
# Cell 7 - Single-sample predict sanity check (English Cell 106 pattern)
print(f"Raw model output: {qwen_v2_predict_raw(sample)!r}")
print(f"Parsed prediction: {qwen_v2_predict(sample):.2f}K VND")
print(f"Ground truth:      {sample.price:.2f}K VND")
print(f"Absolute error:    {abs(qwen_v2_predict(sample) - sample.price):.2f}K VND")

In [ ]:
# Cell 8 - set_seed(42) + Evaluate + charts (English Cell 106)
set_seed(42)
tester = VnTester(
    predictor=qwen_v2_predict,
    data=test_items,
    title="Qwen3.5-4B V2 Fine-tuned",
    size=EVAL_SIZE,
    workers=1,
)
tester.run()

In [ ]:
# Cell 9 - Save results JSON
import numpy as np
from sklearn.metrics import mean_squared_error, r2_score
from utils.evaluator_vn import _rmsle

mae_k = float(np.mean(tester.errors))
mse   = float(mean_squared_error(tester.truths, tester.guesses))
r2    = float(r2_score(tester.truths, tester.guesses))
rmsle = _rmsle(tester.truths, tester.guesses)

results = {
    "model": HUB_MODEL_ID,
    "checkpoint": BEST_CKPT,
    "eval_size": EVAL_SIZE,
    "mae_k_vnd": round(mae_k, 2),
    "mae_vnd": round(mae_k * 1000, 0),
    "rmsle": round(rmsle, 4),
    "mse": round(mse, 2),
    "r2": round(r2, 4),
}
os.makedirs("results", exist_ok=True)
with open(RESULTS_PATH, "w") as f:
    json.dump(results, f, indent=2, ensure_ascii=False)

print(json.dumps(results, indent=2))